In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Import the AnimalShelter CRUD class created in Project One.
# This class is used to connect the dashboard to MongoDB.
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Store the MongoDB username and password used to connect
# to the AAC database through the CRUD module.
username = "aacuser"
password = "Tango2011"

# Create an instance of the AnimalShelter class to establish
# the connection between the dashboard and MongoDB.
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load and encode the Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Define the dashboard layout, including the logo, title,
# rescue filters, data table, pie chart, and map.

app.layout = html.Div([
    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '120px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ),
    
    html.Center(
        html.B(
            html.H1('CS-340 Dashboard - Joey Quinata')
        )
    ),
    
    html.Hr(),
    
    html.Div(
# Provide filtering options for each rescue category.
# Reset returns the dashboard to the complete unfiltered dataset.
        [
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Water Rescue', 'value': 'Water'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                    {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                    {'label': 'Reset', 'value': 'Reset'}
                ],
                value='Reset',
                labelStyle={
                    'display': 'inline-block', 
                    'margin-right': '20px'
                }
            )
        ]
    ),
                    
    html.Hr(),
    
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {
                "name": i, 
                "id": i, 
                "deletable": False,
                "selectable": True
            } 
            for i in df.columns
        ],
        data=df.to_dict('records'),
        row_selectable='single',
        selected_rows=[0],
        page_size=10,
        sort_action='native',
        filter_action='native'
    ),

    html.Br(),
    html.Hr(),
    
    
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            
            html.Div(
                id='map-id',
                className='col s12 m6'
            )
        ]
    )
])
#############################################
# Interaction Between Components / Controller
#############################################


# Display the animal records in an interactive table.
# Users can select one row, sort columns, filter data,
# and move through the results using pagination.
    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        query = {}

    data = db.read(query)
    dff = pd.DataFrame.from_records(data)

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')
                

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    
    # Count each breed and keep the 10 most common breeds
    breed_counts = dff['breed'].value_counts().head(10).reset_index()
    breed_counts.columns = ['breed', 'count']


    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                names='breed',
                values='count',
                title='Top 10 Breed Distribution'
            )
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]
        
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]
    
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://bingoanvil-demosouth-3000.codio.io/proxy/8050/
